# Demo Agent Integration
This notebook demonstrates using a reservoir as a local MCP prefilter and notifying a local Ollama instance.
- Loads a saved ESN model
- Produces embeddings for sliding windows
- Pushes contexts to local MCP store via `mcp_adapter.py`
- Simulates an agent that asks Ollama to reason about a flagged context

1) Setup and imports

In [10]:
# Setup and imports
import os
import numpy as np
import time
import json
import requests
from dotenv import load_dotenv
from data_utils import load_close_matrix, train_val_test_split, windowed_dataset
from mcp_adapter import MCPAdapter, simple_anomaly_score, load_readout_and_embedder
from logger import get_logger

# Initialize logger
logger = get_logger("demo_agent_integration")

# Load environment variables
load_dotenv()
env = os.environ

logger.info("Starting Demo Agent Integration")

# directories
MODELS_DIR = os.path.join(os.getcwd(), "models")
os.makedirs(MODELS_DIR, exist_ok=True)
logger.info(f"Models directory: {MODELS_DIR}")

# choose model artifact path (created by esn notebook)
ESN_MODEL_PATH = os.path.join(MODELS_DIR, "esn_model.pkl")
if not os.path.exists(ESN_MODEL_PATH):
    logger.error("ESN model not found. Run esn_reservoir.ipynb first and save esn_model.pkl")
    raise FileNotFoundError("ESN model not found. Run esn_reservoir.ipynb first and save esn_model.pkl")

logger.info(f"ESN model path: {ESN_MODEL_PATH}")

# Ollama configuration
OLLAMA_CHAT_URL = env.get("OLLAMA_CHAT_URL")
OLLAMA_MODEL = env.get("OLLAMA_MODEL")

logger.info(f"Ollama URL: {OLLAMA_CHAT_URL}")
logger.info(f"Ollama model: {OLLAMA_MODEL}")

2026-03-23 16:59:20,244 - demo_agent_integration - INFO - Starting Demo Agent Integration
2026-03-23 16:59:20,247 - demo_agent_integration - INFO - Models directory: a:\EDUCATION\reservoir_computing\models
2026-03-23 16:59:20,249 - demo_agent_integration - INFO - ESN model path: a:\EDUCATION\reservoir_computing\models\esn_model.pkl
2026-03-23 16:59:20,251 - demo_agent_integration - INFO - Ollama URL: http://localhost:11434/api/chat
2026-03-23 16:59:20,253 - demo_agent_integration - INFO - Ollama model: gemma3:latest


2) Load data and prepare windows

In [11]:
# Load data and prepare windows
logger.info("Loading data and preparing windows...")

df = load_close_matrix()
if df.empty:
    logger.error("No ticker CSVs found in ./tickers. Run ticker.py first.")
    raise RuntimeError("No ticker CSVs found in ./tickers. Run ticker.py first.")

series = df.iloc[:, 0].astype(float).dropna()
logger.info(f"Using ticker: {series.name}, length: {len(series)}")

train, val, test = train_val_test_split(series, train_frac=0.7, val_frac=0.15)
window_size = 10
# We'll run the demo on the validation+test concatenation for variety
demo_series = np.concatenate([val, test])
X_demo, y_demo = windowed_dataset(demo_series, window_size)

print("Demo windows:", X_demo.shape)
logger.info(f"Demo windows prepared: {X_demo.shape}")
logger.info(f"Window size: {window_size}")
logger.info(f"Demo series length: {len(demo_series)}")

2026-03-23 16:59:25,460 - demo_agent_integration - INFO - Loading data and preparing windows...
2026-03-23 16:59:25,464 - data_utils - INFO - Found 10 ticker CSV files
2026-03-23 16:59:25,466 - data_utils - INFO - Loading data from 10 files...
2026-03-23 16:59:25,560 - data_utils - INFO - Created merged matrix with shape (255, 10)
2026-03-23 16:59:25,562 - demo_agent_integration - INFO - Using ticker: ARLO, length: 255
2026-03-23 16:59:25,563 - demo_agent_integration - INFO - Demo windows prepared: (67, 10)
2026-03-23 16:59:25,565 - demo_agent_integration - INFO - Window size: 10
2026-03-23 16:59:25,566 - demo_agent_integration - INFO - Demo series length: 77


Demo windows: (67, 10)


3) Load embedder and adapter

In [12]:
# Load embedder and adapter
logger.info("Loading embedder and adapter...")

embed_fn, readout = load_readout_and_embedder(ESN_MODEL_PATH)
adapter = MCPAdapter(ollama_notify=True)  # set ollama_notify=False to skip local Ollama calls

logger.info("Embedder and adapter loaded successfully")
logger.info(f"Adapter Ollama notify: {adapter.ollama_notify}")

2026-03-23 16:59:28,517 - demo_agent_integration - INFO - Loading embedder and adapter...
2026-03-23 16:59:28,546 - mcp_adapter - INFO - Loaded reservoir model with window_size: 10
2026-03-23 16:59:28,549 - demo_agent_integration - INFO - Embedder and adapter loaded successfully
2026-03-23 16:59:28,552 - demo_agent_integration - INFO - Adapter Ollama notify: True


4) Run a small loop that embeds windows, scores them, and pushes high-score contexts

In [4]:
# Run a small loop that embeds windows, scores them, and pushes high-score contexts
logger.info("Starting window processing loop...")

THRESHOLD = 0.6   # demo threshold for "interesting" windows
flagged = []

logger.info(f"Processing {min(200, len(X_demo))} windows with threshold {THRESHOLD}")

for i, w in enumerate(X_demo[:200]):  # limit to first 200 windows for demo speed
    emb = embed_fn(w)
    score = simple_anomaly_score(emb)
    if score > THRESHOLD:
        meta = adapter.push_context(raw_window=w, embedding=emb, score=score, tag="flagged-window")
        flagged.append(meta)
        print(f"[FLAGGED] idx={i} id={meta['id']} score={score:.3f} path={meta['path']}")
        logger.info(f"[FLAGGED] idx={i} id={meta['id']} score={score:.3f} path={meta['path']}")
    # small throttle for demo
    time.sleep(0.01)
    
    # Log progress every 50 windows
    if (i + 1) % 50 == 0:
        logger.info(f"Processed {i + 1}/{min(200, len(X_demo))} windows")

print("Total flagged:", len(flagged))
logger.info(f"Processing complete. Total flagged: {len(flagged)}")

2026-03-23 15:44:40,388 - demo_agent_integration - INFO - Starting window processing loop...
2026-03-23 15:44:40,392 - demo_agent_integration - INFO - Processing 67 windows with threshold 0.6
2026-03-23 15:44:46,451 - demo_agent_integration - INFO - [FLAGGED] idx=0 id=ctx-0e2680386a20 score=0.721 path=a:\EDUCATION\reservoir_computing\mcp_store\ctx-0e2680386a20.json


[FLAGGED] idx=0 id=ctx-0e2680386a20 score=0.721 path=a:\EDUCATION\reservoir_computing\mcp_store\ctx-0e2680386a20.json


2026-03-23 15:44:52,520 - demo_agent_integration - INFO - [FLAGGED] idx=1 id=ctx-715bb4a5a2ad score=0.721 path=a:\EDUCATION\reservoir_computing\mcp_store\ctx-715bb4a5a2ad.json


[FLAGGED] idx=1 id=ctx-715bb4a5a2ad score=0.721 path=a:\EDUCATION\reservoir_computing\mcp_store\ctx-715bb4a5a2ad.json


2026-03-23 15:44:58,567 - demo_agent_integration - INFO - [FLAGGED] idx=2 id=ctx-3143b4b7d9c6 score=0.721 path=a:\EDUCATION\reservoir_computing\mcp_store\ctx-3143b4b7d9c6.json


...

[FLAGGED] idx=64 id=ctx-30a0471336bf score=0.720 path=a:\EDUCATION\reservoir_computing\mcp_store\ctx-30a0471336bf.json


2026-03-23 16:52:50,380 - demo_agent_integration - INFO - [FLAGGED] idx=65 id=ctx-9b37772baafa score=0.720 path=a:\EDUCATION\reservoir_computing\mcp_store\ctx-9b37772baafa.json


[FLAGGED] idx=65 id=ctx-9b37772baafa score=0.720 path=a:\EDUCATION\reservoir_computing\mcp_store\ctx-9b37772baafa.json


2026-03-23 16:53:49,444 - demo_agent_integration - INFO - [FLAGGED] idx=66 id=ctx-a0d6959e74f5 score=0.720 path=a:\EDUCATION\reservoir_computing\mcp_store\ctx-a0d6959e74f5.json
2026-03-23 16:53:49,462 - demo_agent_integration - INFO - Processing complete. Total flagged: 67


[FLAGGED] idx=66 id=ctx-a0d6959e74f5 score=0.720 path=a:\EDUCATION\reservoir_computing\mcp_store\ctx-a0d6959e74f5.json
Total flagged: 67


5) Simulate an agent invoking Ollama to reason about a flagged context

In [13]:
# We'll pick the first flagged context and ask Ollama to summarize the window stats.

if not flagged:
    logger.warning("No flagged contexts found. Lower THRESHOLD or run on more windows.")
    print("No flagged contexts found. Lower THRESHOLD or run on more windows.")
else:
    logger.info("Simulating agent reasoning with Ollama...")
    ctx = flagged[0]
    ctx_id = ctx["id"]
    ctx_path = ctx["path"]
    
    logger.info(f"Loading context {ctx_id} from {ctx_path}")
    with open(ctx_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    # Build a short prompt that references the context id and summary stats
    prompt = (
        f"Context ID: {ctx_id}\n"
        f"Tag: {payload.get('tag')}\n"
        f"Score: {payload.get('score'):.4f}\n"
        f"Window length: {payload['window_summary']['length']}\n"
        f"Window mean: {payload['window_summary']['mean']:.4f}\n"
        f"Window std: {payload['window_summary']['std']:.4f}\n\n"
        "Please provide a short, actionable summary of what this window might indicate for a short-term trading agent. "
        "Keep it concise and list one suggested next step."
    )
    
    logger.info(f"Sending prompt to Ollama model {OLLAMA_MODEL}")
    
    # Send to local Ollama
    payload_req = {"model": OLLAMA_MODEL, "messages": [{"role": "user", "content": prompt}]}
    try:
        resp = requests.post(OLLAMA_CHAT_URL, json=payload_req, timeout=10)
        resp.raise_for_status()
        result = resp.json()
        logger.info("Received response from Ollama")
        print("Ollama response (truncated):")
        # The exact response shape depends on your Ollama build; print safely
        print(json.dumps(result, indent=2)[:2000])
    except Exception as e:
        logger.error(f"Failed to contact local Ollama: {e}")
        print("Failed to contact local Ollama:", e)

2026-03-23 16:59:36,381 - demo_agent_integration - INFO - Simulating agent reasoning with Ollama...
2026-03-23 16:59:36,384 - demo_agent_integration - INFO - Loading context ctx-0e2680386a20 from a:\EDUCATION\reservoir_computing\mcp_store\ctx-0e2680386a20.json
2026-03-23 16:59:36,386 - demo_agent_integration - INFO - Sending prompt to Ollama model gemma3:latest
2026-03-23 16:59:53,711 - demo_agent_integration - ERROR - Failed to contact local Ollama: Extra data: line 2 column 1 (char 129)


Failed to contact local Ollama: Extra data: line 2 column 1 (char 129)


6) Wrap-up and next steps

In [14]:
# Wrap-up and next steps
logger.info("Demo completed successfully")

print("Demo complete.")
print("Next steps you can try:")
print("- Lower THRESHOLD to flag more windows")
print("- Replace simple_anomaly_score with a learned readout")
print("- Use adapter.ollama_notify=False to run fully offline")
print("- Extend the agent to fetch the full embedding and pass it as structured context to Ollama")

logger.info("Demo Agent Integration finished")

2026-03-23 16:59:53,735 - demo_agent_integration - INFO - Demo completed successfully
2026-03-23 16:59:53,741 - demo_agent_integration - INFO - Demo Agent Integration finished


Demo complete.
Next steps you can try:
- Lower THRESHOLD to flag more windows
- Replace simple_anomaly_score with a learned readout
- Use adapter.ollama_notify=False to run fully offline
- Extend the agent to fetch the full embedding and pass it as structured context to Ollama
